In [ ]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192")

store = {}

# InMemoryChatMessageHistory would store messages history as user and AI for tempory, you can replace it with any other implementation of ChatMessageHistory to store the history in a database or file system.
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"), # this part injects the history of convo from RunnableWithMessageHistory into the prompt, so the model can have the context of previous messages in the conversation.
    ("human", "{input}")
])

chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history, 
    input_messages_key="input",
    history_messages_key="history"
)

while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    res = chain_with_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": "user1"}}
    )
    
    print("AI:", res.content)

In [ ]:
## Stateless RAG

from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

# 1. Load LLM (Groq)
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

# 2. Load Documents
loader = TextLoader("data.txt")  # your file
documents = loader.load()

# 3. Split Documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.split_documents(documents)

# 4. Convert to Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 5. Store in Vector DB (FAISS)
vectorstore = FAISS.from_documents(docs, embeddings)

# 6. Retriever
retriever = vectorstore.as_retriever()

# 7. Prompt Template (IMPORTANT 🔥)
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer ONLY from the provided context.
If the answer is not in context, say "I don't know".

Context:
{context}

Question:
{question}
""")

# 8. RAG Pipeline (NO MEMORY)
def rag_pipeline(question):
    # Retrieve docs
    retrieved_docs = retriever.get_relevant_documents(question)
    
    # Combine context
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Create prompt
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    
    # LLM call
    response = llm.invoke(final_prompt)
    
    return response.content

# 9. Run
while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    answer = rag_pipeline(q)
    
    print("AI:", answer)

In [1]:
print("dh")

dh


In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import MessagesPlaceholder,ChatMessagePromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

llm = ChatGroq(model="llama3-8b-8192", temperature=0)

loader = TextLoader('docs.txt')
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
doc = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(doc,embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

chat_prompt = ChatPromptTemplate(
    ("System","Answer from the given context")
    MessagesPlaceholder(variable_name="history")
    ("human","context: {context}\n\n, Question: {Question}\n\n")
)

store={}

def get_session_id(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    
    return store[session_id]

def retrieve(inputs):
    question = inputs["Question"]
    docs = retriever._get_relevant_documents(question)
    content = "\n\n".join(doc.page_document for doc in doc)
    return {"context":{content} ,"Question":{question}}

chain = retriever|prompt|llm

chat = RunnableWithMessageHistory(
    chain,
    get_session_history,
    history_messages_key="history",
    input_messages_key="input"
)


    





In [ ]:
# Statefull RAG

from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. LLM
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

# 2. Load Docs
loader = TextLoader("data.txt")
documents = loader.load()

# 3. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)

# 4. Embeddings + Vector DB
embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

# 5. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # it use cosine similarity by default, you can change it to other similarity metrics supported by FAISS if needed.

# 6. Prompt (IMPORTANT 🔥)
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful AI assistant. Answer using the provided context. "
     "If not found, say 'I don't know'."),
    
    MessagesPlaceholder(variable_name="history"),
    
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

# 7. RAG Function inside LCEL
def retrieve_context(inputs):
    question = inputs["question"]
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    return {"context": context, "question": question}

# 8. Chain
chain = (
    retrieve_context
    | prompt
    | llm
)

# 9. Memory Store (multi-user)
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# 10. Add Memory
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

# 11. Run
while True:
    q = input("You: ")
    
    if q.lower() in ["exit", "quit"]:
        break
    
    res = chain_with_memory.invoke(
        {"question": q},
        config={"configurable": {"session_id": "user1"}}
    )
    
    print("AI:", res.content)

In [ ]:
# =========================================================
# 1. ENV
# =========================================================

from dotenv import load_dotenv
load_dotenv()

# =========================================================
# 2. LLM
# =========================================================

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

# =========================================================
# 3. DOCUMENT LOADING
# =========================================================

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data.txt")
documents = loader.load()

# =========================================================
# 4. CHUNKING
# =========================================================

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

# =========================================================
# 5. EMBEDDING MODEL
# =========================================================

from langchain.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en",
    encode_kwargs={
        "normalize_embeddings": True # better for cosine similarity calculation
    }
)

# =========================================================
# 6. VECTOR DB (QDRANT + HNSW) (by ANN)
# =========================================================

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from langchain_qdrant import QdrantVectorStore

client = QdrantClient(
    url="http://localhost:6333" 
)

# Create collection
client.recreate_collection(
    collection_name="rag_collection",

    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    )
)

# Store documents
vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
    collection_name="rag_collection"
)

# =========================================================
# 7. RETRIEVER (MMR FOR CONTEXT PACKING)
# =========================================================

retriever = vectorstore.as_retriever(
    search_type="mmr", # context packing strategy pick only top non redudant data

    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

# =========================================================
# 8. QUERY EXPANSION (Multi query generation)
# =========================================================

# from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

from langchain_core.prompts import ChatPromptTemplate

query_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate 3 different versions of the user's question.

Original question:
{question}

Return only the 3 alternative questions, one per line.""")
])



llm_chain = LLMChain(
    llm=llm,
    prompt=query_prompt
)

multi_query_retriever = MultiQueryRetriever(
    retriever=retriever,
    llm_chain=llm_chain
)

# =========================================================
# 9. RERANKER
# =========================================================

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)


# =========================================================
# 10. RETRIEVAL + RERANKING PIPELINE
# =========================================================

def retrieve_context(inputs):

    question = inputs["question"]

    # ---------------------------------------------
    # QUERY EXPANSION + RETRIEVAL
    # ---------------------------------------------

    retrieved_docs = multi_query_retriever.invoke(question)

    # ---------------------------------------------
    # RERANKING
    # ---------------------------------------------

    pairs = [
        [question, doc.page_content]
        for doc in retrieved_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # ---------------------------------------------
    # CONTEXT PACKING
    # ---------------------------------------------

    top_docs = [doc for doc, _ in ranked_docs[:3]]

    context = "\n\n".join([
        doc.page_content
        for doc in top_docs
    ])

    return {
        "context": context,
        "question": question
    }

# =========================================================
# 11. PROMPT
# =========================================================

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

prompt = ChatPromptTemplate.from_messages([

    (
        "system",

        "You are a helpful AI assistant.\n"
        "Answer ONLY from the provided context.\n"
        "If answer is not found, say 'I don't know'."
    ),

    MessagesPlaceholder(variable_name="history"),

    (
        "human",

        "Context:\n{context}\n\n"
        "Question:\n{question}"
    )
])



# =========================================================
# 12. CHAIN
# =========================================================

chain = (
    retrieve_context
    | prompt
    | llm
)

# =========================================================
# 13. MEMORY (KEEPING YOUR SAME LOGIC)
# =========================================================

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,

    input_messages_key="question",
    history_messages_key="history"
)

# =========================================================
# 14. CHAT LOOP
# =========================================================

while True:

    q = input("You: ")

    if q.lower() in ["exit", "quit"]:
        break

    res = chain_with_memory.invoke(

        {"question": q},

        config={
            "configurable": {
                "session_id": "user1"
            }
        }
    )

    print("\nAI:", res.content)

In [7]:
import httpx
import os
from dotenv import load_dotenv  

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

async def call_groq(prompt):
    
    async with httpx.AsyncClient(timeout=30) as client:
        response= await client.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {GROQ_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "openai/gpt-oss-120b",
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            }
        )
        
        if response.status_code == 200:
            
            data = response.json()
            return data["choices"][0]["message"]["content"]
        
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
        
    
result = await call_groq("Hello, how are you?")
print(result)

Hello! I'm doing great, thank you for asking. How can I help you today?


### With Langchain

In [ ]:
import asyncio

from langchain_groq import ChatGroq


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="YOUR_GROQ_API_KEY"
)


async def main():

    results = await asyncio.gather(

        llm.ainvoke("Explain RAG"),

        llm.ainvoke("Explain LLM"),

        llm.ainvoke("Explain embeddings")
    )

    for result in results:
        print(result.content)
        print("-" * 50)


asyncio.run(main())

### Without Langchain We could control everything

In [11]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

async def call_llm(prompt,provider="ollama"):
    
    if provider == 'groq':
        
        base_url= "https://api.groq.com/openai/v1"
        model = "openai/gpt-oss-120b"
        
        headers={
            "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
            "Content-Type": "application/json"
        }
        
    else:
        
        base_url= "https://localhost:11434/v1",
        model= "mistral"
        
        headers={
            "Content-Type": "application/json"
        }
        
    async with httpx.AsyncClient(timeout=30) as client:
            
            response= await client.post(
                f"{base_url}/chat/completions",
                headers=headers,
                json={
                    "model": model,
                    "messages":[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]
                }
            )

            if response.status_code == 200:
                data = response.json()
                return data["choices"][0]["message"]["content"]
            
            else:
                raise Exception(f"Error: {response.status_code}, {response.text}")
            
            

result = await call_llm("Hello, how are you?", provider="groq")
print(result)
        
    

Hello! I'm doing well, thank you for asking. How can I assist you today?


In [ ]:
import os 
import httpx
from dotenv import load_dotenv
load_dotenv()

async def call_llm(prompt,provider="groq"):
    
    if provider == "groq":
        base_url="https://api.groq.com//openai/v1"
        model= "openai/gpt-oss-120b"
        
        headers={
            "Authorization": f"Bearer {os.getenv("GORQ_API_KEY")}",
            "Content-Type": "application/json"
        }

    else:
        base_url="localhost/112437/v1"
        model="mistral"
        
        headers={
            "Content-Type":"application/json"
        }
        
    async with httpx.AsyncClient(timeout=30) as client:
        
        response= await client.post(
            f"{base_url}/chat/compilations",
            headers=headers,
            json={
                "model":model,
                "message":[
                    {
                        'role':'User',
                        'content':prompt
                    }
                ]
            }
        )
        
        if response.status_code == 200:
            data = response.json()
            return data['choices'][0]['message']['content']
        
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
        

### Queue Processing Pattern

In [ ]:
import asyncio

queue = asyncio.Queue() # It creates a asynchronous queue created in the memory

async def producer():
    
    for i in range(5):
        
        await queue.put(i)
        
        print(f"Produced: {i}")

async def consumer():
    
    while True:
        
        item = await queue.get()
        
        print(f"Consumed: {item}")
        
        await asyncio.sleep(1)
        
        queue.task_done() # This would notify the queue that the item has been processed, so it can be removed from the queue.
        
async def main():
    consumer_task= asyncio.create_task(consumer()) # This would run the consumer in the background, so it can consume items from the queue while the producer is producing them.
    
    await producer() # Here producer would start add the jobs to the queue, and the consumer would start consuming them in the background.
    
    print("Producer finished producing items.")
    
    await queue.join() # This would wait until all the items in queue have been consumed, and the consumer has called task_done() for each item.
    
    print("All items have been consumed.")
    
    consumer_task.cancel() # Thiis would stop the consumer once all the jobs consumed to prevent it from running forever, since the consumer is in an infinite loop.
    
asyncio.run(main())

## Queue Processing

#### Producer

have the await queue.put()

#### Consumer

have the await queue.get()

queue.task_done()

#### Main

await asyncio.create_task(consumer())

await queue.join()

consumer.cancel()


In [ ]:
import asyncio
import uuid

queue = asyncio.Queue() 

jobs={}

async def create_job(resume_data):
    
    job_id= str(uuid.uuid4())
    
    jobs[job_id]={
        "resume_data": resume_data,
        "status": "pending"
    }
    
    job={
        "job_id": job_id,
        "resume_data": resume_data,
    }
    
    await queue.put(job)
    
    print(f"Job {job_id} created and added to the queue.")
    
    return job_id

async def worker():
    
    while True:
        
        job= await queue.get()
        
        job_id= job["job_id"]
        resume_data= job["resume_data"]
        
        jobs[job_id]["status"]= "in_progress"
        
        print(f"Processing job {job_id} with resume data: {resume_data}")
        
        # Simulate OCR
        print("  → Extracting text...")
        await asyncio.sleep(2)

        # Simulate LLM
        print("  → Calling LLM...")
        await asyncio.sleep(3)

        # Simulate database
        print("  → Saving result...")
        await asyncio.sleep(1)
        
        jobs[job_id]["status"]= "completed"
        
        print(f"Job {job_id} completed.")
        
        queue.task_done()
        
async def main():
    
    worker_task= asyncio.create_task(worker())
    
    job1 = await create_job("Resume data for John Doe")
    job2 = await create_job("Resume data for Jane Smith")
    
    print(f"Created jobs: {job1}, {job2}")
    
    await queue.join() # Wait until all jobs are processed
    
    print("All jobs have been processed.")
    
    worker_task.cancel() # Stop the worker after all jobs are done
    

asyncio.run(main())

### LLM Concurancy

In [ ]:
import asyncio
import os

from langchain_groq import ChatGroq


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


semaphore = asyncio.Semaphore(3)


async def process(document):

    async with semaphore:

        print(f"Processing: {document}")

        response = await llm.ainvoke(
            f"Extract the important information from:\n{document}"
        )

        print(f"Completed: {document}")

        return response.content


async def main():

    documents = [
        "Resume content 1",
        "Resume content 2",
        "Resume content 3",
        "Resume content 4",
        "Resume content 5",
        "Resume content 6"
    ]

    results = await asyncio.gather(
        *(process(doc) for doc in documents)
    )

    for result in results:
        print(result)


asyncio.run(main())

In [ ]:
import asyncio
from pathlib import Path

from langchain_groq import ChatGroq


# -----------------------------------
# 1. LLM
# -----------------------------------

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="YOUR_GROQ_API_KEY"
)


# -----------------------------------
# 2. Limit concurrency
# -----------------------------------

semaphore = asyncio.Semaphore(3)


# -----------------------------------
# 3. Read documents
# -----------------------------------

def read_documents():

    documents = []

    for file in Path("documents").glob("*.txt"):

        text = file.read_text(
            encoding="utf-8"
        )

        documents.append({
            "name": file.name,
            "text": text
        })

    return documents


# -----------------------------------
# 4. Ask LLM
# -----------------------------------

async def ask_llm(prompt):

    async with semaphore:

        response = await llm.ainvoke(prompt)

        return response.content


# -----------------------------------
# 5. Process one document
# -----------------------------------

async def process_document(document):

    prompt = f"""
    Analyze this document.

    Extract:

    1. Name
    2. Skills
    3. Experience
    4. Summary

    Document:

    {document["text"]}
    """

    result = await ask_llm(prompt)

    return {
        "document": document["name"],
        "result": result
    }


# -----------------------------------
# 6. Process multiple documents
# -----------------------------------

async def process_documents():

    documents = read_documents() # here all the documents are read from the "documents" folder and stored in a list of dictionaries, where each dictionary contains the name and text of a document.

    results = await asyncio.gather(
        *(process_document(doc) for doc in documents)
    )

    return results


# -----------------------------------
# 7. Multiple questions
# -----------------------------------

async def answer_questions():

    questions = [
        "What is RAG?",
        "What are embeddings?",
        "What is LangChain?",
        "What is a vector database?"
    ]

    results = await asyncio.gather(
        *(ask_llm(question) for question in questions)
    )

    return list(zip(questions, results))


# -----------------------------------
# 8. Main
# -----------------------------------

async def main():

    print("Processing documents...\n")

    documents = await process_documents()

    for result in documents:

        print("=" * 50)
        print(result["document"])
        print(result["result"])


    print("\n\nAnswering questions...\n")

    questions = await answer_questions()

    for question, answer in questions:

        print("=" * 50)
        print("QUESTION:", question)
        print("ANSWER:", answer)


asyncio.run(main())

### LLM call with HTTPX 

In [ ]:
import asyncio
import os
import json
from pathlib import Path

import httpx
from dotenv import load_dotenv


# ============================================================
# CONFIGURATION
# ============================================================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Choose:
# "groq"
# "ollama"

LLM_PROVIDER = "groq"


# ============================================================
# LLM CONFIGURATION
# ============================================================

if LLM_PROVIDER == "groq":

    BASE_URL = "https://api.groq.com/openai/v1"

    MODEL = "llama-3.3-70b-versatile"

    HEADERS = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

else:

    BASE_URL = "http://localhost:11434/v1"

    MODEL = "mistral"

    HEADERS = {
        "Content-Type": "application/json"
    }


# ============================================================
# CONCURRENCY LIMIT
# ============================================================

# Maximum 3 LLM requests at the same time

semaphore = asyncio.Semaphore(3)


# ============================================================
# ASYNC LLM CALL
# ============================================================

async def call_llm(prompt):

    max_retries = 3

    for attempt in range(max_retries):

        try:

            # Only 3 requests can enter this section
            # at the same time.

            async with semaphore:

                async with httpx.AsyncClient(
                    timeout=60
                ) as client:

                    response = await client.post(

                        f"{BASE_URL}/chat/completions",

                        headers=HEADERS,

                        json={
                            "model": MODEL,

                            "messages": [
                                {
                                    "role": "user",
                                    "content": prompt
                                }
                            ],

                            "temperature": 0
                        }
                    )

                    response.raise_for_status()

                    data = response.json()

                    return data["choices"][0]["message"]["content"]


        except httpx.HTTPError as error:

            print(
                f"LLM request failed "
                f"(attempt {attempt + 1}): {error}"
            )

            # Last attempt
            if attempt == max_retries - 1:
                raise

            # Exponential backoff
            wait_time = 2 ** attempt

            print(
                f"Retrying in {wait_time} seconds..."
            )

            await asyncio.sleep(wait_time)


# ============================================================
# READ DOCUMENTS
# ============================================================

def read_documents():

    documents = []

    document_path = Path("documents")

    for file in document_path.glob("*.txt"):

        text = file.read_text(
            encoding="utf-8"
        )

        documents.append(
            {
                "name": file.name,
                "text": text
            }
        )

    return documents


# ============================================================
# PROCESS ONE DOCUMENT
# ============================================================

async def process_document(document):

    prompt = f"""
You are a resume analyzer.

Analyze the following resume.

Extract:

1. Candidate name
2. Technical skills
3. Experience
4. Education
5. Short summary

Resume:

{document["text"]}
"""

    result = await call_llm(prompt)

    return {
        "document": document["name"],
        "analysis": result
    }


# ============================================================
# PROCESS MULTIPLE DOCUMENTS
# ============================================================

async def process_documents():

    documents = read_documents()

    print(
        f"\nFound {len(documents)} documents."
    )

    print(
        "Processing documents concurrently...\n"
    )

    results = await asyncio.gather(

        *(
            process_document(document)
            for document in documents
        )

    )

    return results


# ============================================================
# MULTIPLE QUESTIONS
# ============================================================

async def answer_questions():

    questions = [

        "What is RAG?",

        "What are embeddings?",

        "What is a vector database?",

        "What is an LLM?",

        "What is the difference between RAG and fine-tuning?"

    ]

    print(
        "\nProcessing multiple questions concurrently...\n"
    )

    results = await asyncio.gather(

        *(
            call_llm(question)
            for question in questions
        )

    )

    return list(
        zip(questions, results)
    )


# ============================================================
# MAIN
# ============================================================

async def main():

    # --------------------------------------------------------
    # PART 1
    # PROCESS DOCUMENTS
    # --------------------------------------------------------

    document_results = await process_documents()

    print("\n")
    print("=" * 60)
    print("DOCUMENT RESULTS")
    print("=" * 60)

    for result in document_results:

        print("\nDocument:")
        print(result["document"])

        print("\nAnalysis:")
        print(result["analysis"])

        print("-" * 60)


    # --------------------------------------------------------
    # PART 2
    # ANSWER QUESTIONS
    # --------------------------------------------------------

    question_results = await answer_questions()

    print("\n")
    print("=" * 60)
    print("QUESTION RESULTS")
    print("=" * 60)

    for question, answer in question_results:

        print("\nQuestion:")
        print(question)

        print("\nAnswer:")
        print(answer)

        print("-" * 60)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    asyncio.run(main())

In [ ]:
"""

18. How would you prevent duplicate document processing?
Use a unique document/job ID, usually based on something like:
- document ID
- file hash
- request ID
For example:
file_hash = sha256(file_bytes).hexdigest()


Store it in the database.
Before processing:
Hash already exists?
       │
   ┌───┴───┐
  YES      NO
   │        │
Skip      Process

Also make the database operation idempotent where possible.


19. How would you handle LLM rate limits?
If the provider returns something like:
429 Too Many Requests

I would:
1. Limit concurrency with a semaphore.
2. Implement retry logic.
3. Use exponential backoff.
4. Respect provider-provided Retry-After when available.
5. Control request/token rate if necessary.
6. Use a queue to smooth traffic.
Architecture:
1000 requests
      ↓
Queue
      ↓
Workers
      ↓
Concurrency limit
      ↓
Rate control
      ↓
LLM

Important distinction:
Semaphore controls concurrent requests; rate limiting controls how many requests/tokens are sent over time.

"""


## Complete Langchain RAG

In [ ]:
import asyncio
import json
import os

import httpx

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import Chroma


# ============================================================
# CONFIGURATION
# ============================================================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

BASE_URL = "https://api.groq.com/openai/v1"

MODEL = "llama-3.3-70b-versatile"

MAX_CONCURRENT_REQUESTS = 3

MAX_RETRIES = 3


HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}


# ============================================================
# PYDANTIC RESPONSE
# ============================================================

class RAGResponse(BaseModel):

    answer: str

    sources: list[str]


# ============================================================
# LOAD DOCUMENTS USING LANGCHAIN
# ============================================================

def load_documents():

    loader = DirectoryLoader(
        "documents",
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={
            "encoding": "utf-8"
        }
    )

    documents = loader.load()

    print(
        f"Loaded {len(documents)} documents"
    )

    return documents


# ============================================================
# SPLIT DOCUMENTS USING LANGCHAIN
# ============================================================

def split_documents(documents):

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=500,

        chunk_overlap=50
    )

    chunks = splitter.split_documents(
        documents
    )

    print(
        f"Created {len(chunks)} chunks"
    )

    return chunks


# ============================================================
# CREATE EMBEDDINGS
# ============================================================

def create_embeddings():

    embeddings = HuggingFaceEmbeddings(

        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings


# ============================================================
# CREATE VECTOR DATABASE
# ============================================================

def create_vector_db(
    chunks,
    embeddings
):

    vector_db = Chroma.from_documents(

        documents=chunks,

        embedding=embeddings,

        persist_directory="./chroma_db"
    )

    print("Vector database created")

    return vector_db


# ============================================================
# CREATE RETRIEVER
# ============================================================

def create_retriever(vector_db):

    retriever = vector_db.as_retriever(

        search_type="similarity",

        search_kwargs={
            "k": 3
        }
    )

    return retriever


# ============================================================
# BUILD RAG PROMPT
# ============================================================

def build_prompt(
    question,
    documents
):

    context = "\n\n".join(

        f"Source: {doc.metadata.get('source')}\n"
        f"{doc.page_content}"

        for doc in documents
    )

    prompt = f"""
You are a RAG assistant.

Answer the question using ONLY the provided context.

If the answer cannot be found in the context,
say that the information is not available.

Return ONLY valid JSON:

{{
    "answer": "your answer",
    "sources": ["source1.txt"]
}}

Context:

{context}

Question:

{question}
"""

    return prompt


# ============================================================
# HTTPX ASYNC LLM CALL
# ============================================================

async def call_llm(
    client,
    prompt
):

    for attempt in range(MAX_RETRIES):

        try:

            response = await client.post(

                f"{BASE_URL}/chat/completions",

                headers=HEADERS,

                json={

                    "model": MODEL,

                    "messages": [

                        {
                            "role": "user",
                            "content": prompt
                        }

                    ],

                    "temperature": 0
                }
            )

            response.raise_for_status()

            data = response.json()

            return (
                data["choices"][0]
                ["message"]
                ["content"]
            )


        except httpx.HTTPError as error:

            print(
                f"HTTP error "
                f"(attempt {attempt + 1}): "
                f"{error}"
            )

            if attempt == MAX_RETRIES - 1:

                raise

            wait_time = 2 ** attempt

            print(
                f"Retrying in "
                f"{wait_time} seconds..."
            )

            await asyncio.sleep(
                wait_time
            )


# ============================================================
# ASK ONE QUESTION
# ============================================================

async def ask_rag(
    client,
    retriever,
    question,
    semaphore
):

    # --------------------------------------------------------
    # Retrieval
    # --------------------------------------------------------

    documents = await retriever.ainvoke(
        question
    )

    print(
        f"\nRetrieved {len(documents)} chunks "
        f"for: {question}"
    )

    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    prompt = build_prompt(
        question,
        documents
    )

    # --------------------------------------------------------
    # Limit concurrent LLM calls
    # --------------------------------------------------------

    async with semaphore:

        response = await call_llm(
            client,
            prompt
        )

    # --------------------------------------------------------
    # Parse JSON
    # --------------------------------------------------------

    data = json.loads(
        response
    )

    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    result = RAGResponse.model_validate(
        data
    )

    return result


# ============================================================
# MULTIPLE QUESTIONS
# ============================================================

async def process_questions(
    questions,
    retriever
):

    semaphore = asyncio.Semaphore(
        MAX_CONCURRENT_REQUESTS
    )

    async with httpx.AsyncClient(
        timeout=60
    ) as client:

        tasks = [

            ask_rag(
                client,
                retriever,
                question,
                semaphore
            )

            for question in questions
        ]

        results = await asyncio.gather(
            *tasks
        )

    return results


# ============================================================
# MAIN
# ============================================================

async def main():

    print("=" * 60)

    print(
        "ASYNC RAG"
    )

    print(
        "LangChain Retrieval + HTTPX LLM"
    )

    print("=" * 60)


    # ========================================================
    # STEP 1
    # ========================================================

    documents = load_documents()


    # ========================================================
    # STEP 2
    # ========================================================

    chunks = split_documents(
        documents
    )


    # ========================================================
    # STEP 3
    # ========================================================

    embeddings = create_embeddings()


    # ========================================================
    # STEP 4
    # ========================================================

    vector_db = create_vector_db(
        chunks,
        embeddings
    )


    # ========================================================
    # STEP 5
    # ========================================================

    retriever = create_retriever(
        vector_db
    )


    # ========================================================
    # STEP 6
    # ========================================================

    questions = [

        "What is RAG?",

        "What is Python used for?",

        "What is supervised learning?",

        "What are some machine learning algorithms?",

        "What are embeddings?"

    ]


    # ========================================================
    # STEP 7
    # ========================================================

    results = await process_questions(

        questions,

        retriever
    )


    # ========================================================
    # STEP 8
    # ========================================================

    print("\n")
    print("=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)


    for question, result in zip(
        questions,
        results
    ):

        print("\nQUESTION:")
        print(question)

        print("\nANSWER:")
        print(result.answer)

        print("\nSOURCES:")
        print(result.sources)

        print("-" * 60)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    asyncio.run(main())